# Banana Leaf Disease Classifier (BananaLSD)
EfficientNetB0 transfer learning · **OriginalSet only** (no augmented-set leakage) · stratified split · 2-stage fine-tuning · TTA

**Runtime → Change runtime type → T4 GPU**, then run all cells.

## 1. Setup

In [ ]:
import os, glob, shutil, hashlib, random, zipfile, json
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Upload the dataset zip from your device & extract OriginalSet only

In [ ]:
from google.colab import files

EXTRACT_TO = "/content/data"
CLASS_NAMES = {"cordana", "healthy", "pestalotiopsis", "sigatoka"}

# --- pick the uploaded zip (re-use one already in /content, else open upload dialog) ---
zips = [z for z in glob.glob("/content/*.zip") if os.path.getsize(z) > 5e6]
if not zips:
    up = files.upload()
    zips = [f"/content/{n}" for n in up if n.lower().endswith(".zip")]
assert zips, "No valid zip found. Upload the BananaLSD dataset zip."
ZIP_PATH = max(zips, key=os.path.getsize)
print(f"Zip: {os.path.basename(ZIP_PATH)} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")

# --- what does the zip contain? ---
with zipfile.ZipFile(ZIP_PATH) as z:
    names = [n for n in z.namelist() if not n.endswith("/")]
sets = {}
for n in names:
    low = n.lower()
    key = "augmented" if "augment" in low else ("original" if "original" in low else "other")
    sets[key] = sets.get(key, 0) + 1
print("Files in zip by set:", sets)
assert sets.get("original", 0) > 0, (
    "This zip has no OriginalSet folder. Zip the OriginalSet folder from your PC and upload that instead.")

# --- extract ONLY original files (augmented is never touched) ---
shutil.rmtree(EXTRACT_TO, ignore_errors=True)
os.makedirs(EXTRACT_TO, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    members = [m for m in z.namelist() if "augment" not in m.lower() and "original" in m.lower()]
    z.extractall(EXTRACT_TO, members)

# --- find the folder that directly holds the 4 class folders ---
cands = []
for d in glob.glob(f"{EXTRACT_TO}/**/", recursive=True):
    d = d.rstrip("/")
    subs = {s.lower() for s in os.listdir(d) if os.path.isdir(os.path.join(d, s))}
    if CLASS_NAMES <= subs and "augment" not in d.lower():
        cands.append(d)
assert cands, "Could not find the 4 class folders inside OriginalSet - paste the output of the listing below."
DATA_DIR = cands[0]
print("\nUsing:", DATA_DIR)
for c in sorted(os.listdir(DATA_DIR)):
    print(f"  {c:16s} {len(os.listdir(os.path.join(DATA_DIR, c)))}")

## 3. File list, duplicate removal, stratified split (70/15/15)

In [ ]:
CLASSES = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
print("Classes:", CLASSES)

rows = []
for c in CLASSES:
    for f in glob.glob(os.path.join(DATA_DIR, c, "*")):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            rows.append((f, class_to_idx[c]))
df = pd.DataFrame(rows, columns=["path", "label"])

# drop exact duplicate files so one photo can't land in both train and test
df["md5"] = df.path.map(lambda p: hashlib.md5(open(p, "rb").read()).hexdigest())
before = len(df)
df = df.drop_duplicates("md5").reset_index(drop=True)
print(f"Images: {before} -> {len(df)} after removing exact duplicates")
print(df.label.map(lambda i: CLASSES[i]).value_counts())

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df.label, random_state=SEED)
print("train/val/test:", len(train_df), len(val_df), len(test_df))

# class weights (Sigatoka has far more images)
counts = np.bincount(train_df.label, minlength=len(CLASSES))
cw = (counts.sum() / (len(CLASSES) * counts)).astype("float32")
print("Class weights:", dict(zip(CLASSES, cw.round(2))))

## 4. Input pipeline (augmentation on TRAIN only)

In [ ]:
IMG = 224
BATCH = 32
NUM_CLASSES = len(CLASSES)
cw_tensor = tf.constant(cw)

def load(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG, IMG))            # EfficientNet expects 0-255 floats
    return img, tf.one_hot(label, NUM_CLASSES), tf.gather(cw_tensor, label)

augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
], name="augment")

def make_ds(d, train=False):
    ds = tf.data.Dataset.from_tensor_slices((d.path.values, d.label.values.astype("int32")))
    if train:
        ds = ds.shuffle(len(d), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH)
    if train:
        ds = ds.map(lambda x, y, w: (augment(x, training=True), y, w), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df), make_ds(test_df)

x, y, _ = next(iter(train_ds))
plt.figure(figsize=(10, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1); plt.imshow(x[i].numpy().astype("uint8"))
    plt.title(CLASSES[int(tf.argmax(y[i]))]); plt.axis("off")
plt.show()

## 5. Model: EfficientNetB0 (ImageNet pretrained)

In [ ]:
base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(IMG, IMG, 3))
base.trainable = False

inputs = keras.Input((IMG, IMG, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs)

loss = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=loss, metrics=["accuracy"])
model.summary()

## 6. Stage 1 – train the head

In [ ]:
def cbs(path):
    return [
        keras.callbacks.ModelCheckpoint(path, monitor="val_accuracy", save_best_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ]

h1 = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=cbs("stage1.keras"))

## 7. Stage 2 – fine-tune the top of the backbone (low LR, BatchNorm frozen)

In [ ]:
base.trainable = True
for l in base.layers[:-40]:
    l.trainable = False
for l in base.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss=loss, metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=40, callbacks=cbs("best_banana_model.keras"))

acc = h1.history["accuracy"] + h2.history["accuracy"]; vacc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
ls = h1.history["loss"] + h2.history["loss"]; vls = h1.history["val_loss"] + h2.history["val_loss"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(acc, label="train"); ax[0].plot(vacc, label="val"); ax[0].set_title("Accuracy"); ax[0].legend()
ax[1].plot(ls, label="train"); ax[1].plot(vls, label="val"); ax[1].set_title("Loss"); ax[1].legend()
plt.show()

## 8. Final test-set evaluation (flip TTA)

In [ ]:
def predict_tta(ds):
    probs, ys = [], []
    for x, y, _ in ds:
        p = model.predict(x, verbose=0)
        p += model.predict(tf.image.flip_left_right(x), verbose=0)
        p += model.predict(tf.image.flip_up_down(x), verbose=0)
        probs.append(p / 3); ys.append(y.numpy())
    return np.concatenate(probs), np.concatenate(ys).argmax(1)

probs, y_true = predict_tta(test_ds)
y_pred = probs.argmax(1)

print("Test accuracy:", (y_pred == y_true).mean().round(4))
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix (test set)"); plt.show()

## 9. Save model & predict on your own leaf photo

In [ ]:
model.save("banana_leaf_model.keras")
json.dump(CLASSES, open("classes.json", "w"))

def predict_image(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG, IMG))[None]
    p = (model.predict(img, verbose=0) + model.predict(tf.image.flip_left_right(img), verbose=0)) / 2
    i = int(p.argmax())
    plt.imshow(img[0].numpy().astype("uint8")); plt.axis("off")
    plt.title(f"{CLASSES[i]} ({p[0][i]*100:.1f}%)"); plt.show()

# Try it:
# up = files.upload(); predict_image(list(up)[0])
# files.download("banana_leaf_model.keras")